# Fase 1: Business Case & Data Collection


## 1. Business Understanding

El objetivo principal de este proyecto es realizar un análisis estratégico del mercado de alojamientos turísticos en la ciudad de Málaga. A través de la exploración de datos, buscamos identificar los factores críticos que influyen en la variación de precios y detectar patrones de éxito en la oferta actual.

- Problem Statement: "¿En qué zonas y bajo qué condiciones es más competitivo el mercado de Airbnb en Málaga?"

- Stakeholders: Inversores inmobiliarios, gestores de alojamientos turísticos (hosts) y entidades de planificación urbana.

- Métricas de Éxito:

Volumen de reseñas como indicador de demanda/ocupación.

Precio medio por tipo de alojamiento y distrito.

Niveles de profesionalización del sector (proporción de empresas vs. particulares).

## 2. Hipótesis y plan de acción

###  Hipótesis 01: Dominio del Distrito Centro
**Enunciado:** Los alojamientos ubicados en el **Distrito Centro** presentan precios significativamente más altos debido a la concentración masiva de la oferta y la alta demanda turística.

* **Plan de Acción:** 1. Correlacionar ubicación geográfica con precio.
    2. Analizar si el tipo de habitación justifica el sobreprecio en el centro.
    3. Validar si la disponibilidad anual influye en esta zona.

---

###  Hipótesis 02: Relación Precio-Rotación (Efecto Budget)
**Enunciado:** Existe una relación inversa entre el precio y la rotación; los alojamientos con precios por debajo de la media acumulan un volumen de reseñas significativamente mayor.

* **Plan de Acción:** 1. Categorizar los precios en cuartiles (Bajo, Medio, Alto).
    2. Comparar el promedio de reseñas por cada grupo de precio.
    3. Identificar si el segmento de bajo coste es el que sostiene el volumen del mercado.

---

###  Hipótesis 03: Estacionalidad de la Demanda
**Enunciado:** La demanda en Málaga presenta un pico de crecimiento superior al **50%** durante el tercer trimestre (verano), siendo el Centro el que mantiene mayor estabilidad el resto del año.

* **Plan de Acción:** 1. Realizar un `merge` de los datasets de *listings* y *reviews*.
    2. Extraer series temporales (mes/año) de las reseñas.
    3. Comparar la evolución temporal entre distritos de costa y el casco histórico.

## 3. Requerimiento de los datos

En este punto se define el marco técnico de los datos necesarios para validar las hipótesis de negocio planteadas anteriormente.

**3.1. Fuentes de Datos:**  

Para este análisis se utilizarán dos fuentes de datos principales proporcionadas por el portal Inside Airbnb para la ciudad de Málaga:

- listings.csv: Archivo maestro que contiene el inventario detallado de los alojamientos, sus características físicas, de ubicación y precios.

- reviews.csv: Histórico de reseñas que servirá como indicador de la actividad y demanda temporal de cada alojamiento.

**3.2. Granularidad y Ventana Temporal:**  

Granularidad: La unidad mínima de análisis es el alojamiento individual (identificado por su id). Para el análisis geográfico, los datos se agruparán a nivel de barrio (neighbourhood).

Ventana Temporal: * Análisis estático: Se utilizará la "foto fija" actual del dataset para precios y oferta.

Análisis dinámico: Se requiere un histórico de al menos 12 meses de reseñas para detectar picos estacionales (Hipótesis 03) y validar el comportamiento del mercado en diferentes trimestres.

**3.3. Variables Mínimas Imprescindibles:**  

Para que el EDA sea viable y las hipótesis puedan ser validadas o refutadas, se han identificado las siguientes variables como obligatorias:

| Variable | Tipo de Dato | Rol en el Análisis | Justificación Técnica |
| :--- | :--- | :--- | :--- |
| **`id`** | Int64 | **Identificador** | Clave primaria necesaria para el cruce con el dataset de `reviews`. |
| **`neighbourhood`** | Object | **Segmentación** | Variable crítica para el análisis geográfico y validación de la **Hipótesis 1**. |
| **`price`** | Object* | **Objetivo** | Variable dependiente principal. *Requiere limpieza de símbolos ($) y conversión a float.* |
| **`room_type`** | Object | **Predictor** | Categorización del alojamiento para explicar variaciones de precio por tipología. |
| **`host_id`** | Int64 | **Dimensionamiento** | Permite calcular el volumen de propiedades por host (Profesionalización). |
| **`number_of_reviews`** | Int64 | **Proxy Demanda** | Métrica de actividad para validar la **Hipótesis 2** (Relación Precio-Rotación). |
| **`last_review`** | Object (Date) | **Temporalidad** | Define la vigencia del dato y permite segmentar por fecha de última actividad. |


## 4. Disponibilidad

El dato es plenamente disponible. Contamos con tres archivos fuente que se complementan entre sí (mapeo de barrios, listados detallados y comentarios de usuarios).  

No existen barreras técnicas para su acceso ya que los archivos están en formato tabular estándar.  
Se trata de un dataset público (de Inside Airbnb) que contiene la información necesaria para el periodo actual en Málaga.  

Tampoco existen restricciones de acceso, lo cual nos permite continuar.

## 5. Adquisición de datos

Los datos provienen de Inside Airbnb, una plataforma de datos independientes de código abierto que recopila información pública de los anuncios de Airbnb en Málaga.

In [4]:
# Importamos la librería necesaria
import pandas as pd

# Archivo cargado desde su origen, en este caso CSV
df_2025 = pd.read_csv("../../src/data/listings.csv")
df = df_2025.copy()
df.head(2)

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,96033,"Bonito piso a 200m de la playa, El Palo (Málaga)",510467,Rafael,NaN,Este,36.72031,-4.35627,Entire home/apt,58.0,3,274,2025-09-29,1.88,1,324,40,ESFCTU0000290200003588210000000000000000VUT/MA...
1,166473,Perfect Location In Malaga,793360,Fred,NaN,Este,36.72031,-4.36108,Private room,28.0,5,102,2025-03-27,0.59,5,288,3,NaN


## 6. Calidad de los datos

Vamos a definir una función para mostrar la calidad en una tabla organizada.

In [5]:
def calidad_datos(df):
    resumen = pd.DataFrame({
        'dtype': df.dtypes,
        'nulos': df.isnull().sum(),
        'pct_nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'unicos': df.nunique()
    })
    return resumen

print(calidad_datos(df))

                                  dtype  nulos  pct_nulos  unicos
id                                int64      0       0.00    9714
name                                str      0       0.00    9441
host_id                           int64      0       0.00    3504
host_name                           str      0       0.00    1674
neighbourhood_group             float64   9714     100.00       0
neighbourhood                       str      0       0.00      11
latitude                        float64      0       0.00    5559
longitude                       float64      0       0.00    6324
room_type                           str      0       0.00       4
price                           float64    899       9.25     525
minimum_nights                    int64      0       0.00      43
number_of_reviews                 int64      0       0.00     450
last_review                         str   1005      10.35     926
reviews_per_month               float64   1005      10.35     664
calculated

Tras realizar una auditoría técnica preliminar sobre el archivo df_2025.csv, hemos identificado los siguientes puntos críticos que determinan la calidad del proyecto:

1) Integridad y Valores Faltantes (Missing Values): Se observa una ausencia total de datos en la columna neighbourhood_group (100% de nulos). 
Sin embargo, esto no compromete el proyecto ya que la columna neighbourhood está completa y disponemos del archivo maestro de barrios y el mapa para realizar la clasificación geográfica.  
La columna de precios presenta 899 valores nulos, lo que representa un 9.25% del total del dataset. Hemos decidido que, dado que el volumen de nulos es inferior al 10%, procederemos a eliminar esas filas para el análisis de rentabilidad, ya que trabajar con precios estimados (imputados) podría sesgar las conclusiones finales sobre el mercado de Málaga.

2) Por otro lado, variables clave como reviews_per_month y last_review presentan aproximadamente un 20% de valores nulos. Hemos verificado que estos nulos coinciden exactamente con propiedades que tienen 0 reseñas; por tanto, no son errores de carga, sino una característica del negocio (propiedades nuevas o sin actividad).

3) Además, el archivo de reseñas no tiene el precio ni el tipo de vivienda, solo tiene la fecha y el comentario.
Por una parte: Para que "cuadre" y podamos usar las reseñas en nuestras hipótesis, en preciso que hagamos un merge (o unión), con el "puente" que une ambos archivos: la columna listing_id (en reviews) y la columna id (en listings).
Por la otra: Requiere una transformación de la variable date de tipo string a tipo datetime para poder realizar análisis de tendencias estacionales en Málaga.

4) Consistencia y Tipado: Los identificadores (id) son consistentes entre los archivos, lo que garantiza que la unión de las tablas de precios con las de reseñas sea fiable. 

5) Ruido y Outliers: La calidad se ve afectada por valores extremos en el precio y en el número mínimo de noches (minimum_nights). 
Estos "datos ruidosos" podrían sesgar las medias de los barrios de Málaga si no se tratan adecuadamente en la fase de limpieza.  


CONCLUSIÓN DE CALIDAD: A pesar de los valores faltantes en columnas secundarias, el dataset posee un nivel de calidad alto para los objetivos del negocio. 
Los datos son auténticos, provienen de registros reales de la plataforma y el volumen de muestras es lo suficientemente grande como para garantizar significancia estadística en las conclusiones.

## 7. Revisión de Hipótesis de Partida

Para guiar el Análisis Exploratorio de Datos (EDA) en las fases posteriores, se establecen tres hipótesis de negocio fundamentales. Estas hipótesis nacen de la intuición comercial y del entendimiento del mercado turístico en Málaga, y serán validadas o rechazadas mediante el análisis estadístico y la visualización de datos:

### Hipótesis 1: El factor geográfico es el principal determinante del precio
* **Enunciado:** Los alojamientos ubicados en el Distrito Centro presentan un precio mediano significativamente superior al de los distritos periféricos de Málaga.
* **Justificación de negocio:** El Distrito Centro concentra la mayor parte de la oferta cultural, histórica y gastronómica de la ciudad, lo que genera una alta disposición al pago por parte de los turistas que buscan proximidad a los puntos de interés, permitiendo a los hosts fijar tarifas más altas.

### Hipótesis 2: Relación inversa entre veteranía (reseñas) y precio de salida
* **Enunciado:** El precio de los alojamientos con pocas reseñas es más alto que el de aquellos que tienen un volumen elevado de opiniones.
* **Justificación de negocio:** Se asume que los "anfitriones novatos" o con propiedades nuevas tienden a sobrevalorar sus inmuebles por falta de optimización o para testear el mercado. Por el contrario, los anfitriones veteranos (con muchas reseñas) ajustan y optimizan sus precios a la baja para garantizar una ocupación constante.

### Hipótesis 3: Estacionalidad extrema de verano con estabilidad en el Centro
* **Enunciado:** La demanda de alojamientos presenta un pico de crecimiento superior al 50% durante el tercer trimestre del año (Q3: julio, agosto, septiembre), siendo el Distrito Centro el que mantiene la demanda más estable durante el resto de las estaciones frente a las otras zonas.
* **Justificación de negocio:** Málaga es un destino costero tradicionalmente ligado al turismo de sol y playa, por lo que se espera una explosión de reservas en verano. Sin embargo, se intuye que la diversificación cultural del Centro lo protege de la estacionalidad, manteniendo un flujo de caja constante para los inversores durante todo el año.